# FLAN-T5 summarization: full baseline → fine-tuning → full evaluation
Run top to bottom in Kaggle with **Internet enabled** and **one T4 or P100 GPU**. No uploaded repository or dataset is required. All outputs go under `/kaggle/working/training/`.

The same tokenizer, model object, full test rows, four-beam generation, and ROUGE routine are used before and after training. Checkpoint selection uses validation ROUGE-L only. Test scores must not guide checkpoint selection.

Full test generation twice plus full validation generation every 500 optimizer steps can take hours; measure the progress bars rather than assuming a few minutes. Save a Kaggle version with outputs before ending the session. This notebook contains no precomputed training results.

In [ ]:
# Run before importing any of these libraries. If already imported, restart the
# kernel after installation and continue with the next cell.
%pip install --upgrade transformers datasets accelerate evaluate rouge_score sentencepiece nltk pandas matplotlib

In [ ]:
import os
# Keep this notebook on one GPU, even if the selected Kaggle option exposes two.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from collections import Counter
from pathlib import Path
import gc
import hashlib
import importlib.metadata as metadata
import json
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import evaluate
from datasets import DatasetDict, concatenate_datasets, load_dataset
from nltk.tokenize import PunktSentenceTokenizer
from tqdm.auto import tqdm
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback, set_seed)

if not torch.cuda.is_available():
    raise RuntimeError("GPU required. Enable a T4 or P100 in Kaggle notebook Settings > Accelerator, then restart.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
VERSIONS = {name: metadata.version(name) for name in
            ("torch", "transformers", "datasets", "accelerate", "evaluate", "rouge_score", "nltk")}
print(VERSIONS)

ROOT = Path("/kaggle/working")
DATA_DIR = ROOT / "training/data"
CHECKPOINT_DIR = ROOT / "training/checkpoints/flan-t5-summarizer"
BEST_DIR = CHECKPOINT_DIR / "best"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/flan-t5-base"
MAX_INPUT_LENGTH, MAX_TARGET_LENGTH = 512, 128
TASK_PREFIX = "summarize: "
SEED = 42
DATASET_REPOS = {
    "cnn_dailymail": ("abisee/cnn_dailymail", "3.0.0"),
    "xsum": ("EdinburghNLP/xsum",),
    "samsum": ("knkarthick/samsum",),
}
FIELD_MAP = {
    "cnn_dailymail": {"article": "text", "highlights": "summary"},
    "xsum": {"document": "text", "summary": "summary"},
    "samsum": {"dialogue": "text", "summary": "summary"},
}
TRAIN_CAPS = {"cnn_dailymail": 15000, "xsum": 15000, "samsum": 10000}
SPLITS = ("train", "validation", "test")
TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE = 8, 8
GRADIENT_ACCUMULATION_STEPS = 2
USE_FP16 = True
RESUME_FROM_CHECKPOINT = None  # Set to a checkpoint-N directory to resume training.
set_seed(SEED)

def save_json(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(content, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    temporary.replace(path)


## Rebuild Phase 3 data
The field mapping, seed, train-only caps, task prefix, fixed padding, and label masking match the local preprocessing script. Validation and test remain full. Source labels are retained separately **before** the tokenization map removes raw columns. No shuffling occurs after that map.

In [ ]:
def build_unified_dataset(split):
    if split not in SPLITS:
        raise ValueError(split)
    parts = []
    for source, repo_args in DATASET_REPOS.items():
        print(f"Loading {source}/{split}...", flush=True)
        dataset = load_dataset(*repo_args, split=split)
        if split == "train":
            dataset = dataset.shuffle(seed=SEED).select(range(min(TRAIN_CAPS[source], len(dataset))))
        mapping = FIELD_MAP[source]
        dataset = dataset.select_columns(list(mapping))
        dataset = dataset.rename_columns({old: new for old, new in mapping.items() if old != new})
        dataset = dataset.add_column("source", [source] * len(dataset))
        parts.append(dataset)
    return concatenate_datasets(parts).shuffle(seed=SEED)

def tokenize_batch(examples):
    encoded = tokenizer([TASK_PREFIX + text for text in examples["text"]],
                        max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length")
    targets = tokenizer(text_target=examples["summary"], max_length=MAX_TARGET_LENGTH,
                        truncation=True, padding="max_length")
    encoded["labels"] = [[token if token != tokenizer.pad_token_id else -100 for token in row]
                         for row in targets["input_ids"]]
    return encoded

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
unified = DatasetDict({split: build_unified_dataset(split) for split in SPLITS})
print("Train mix:", dict(Counter(unified["train"]["source"])))
assert dict(Counter(unified["train"]["source"])) == TRAIN_CAPS
test_sources = list(unified["test"]["source"])
tokenized = unified.map(tokenize_batch, batched=True,
                        remove_columns=["text", "summary", "source"], desc="Tokenizing")
for split in SPLITS:
    assert len(tokenized[split]) == len(unified[split])
    assert set(tokenized[split].column_names) == {"input_ids", "attention_mask", "labels"}
    print(split, len(tokenized[split]))

# Fingerprint actual ordered test content, not just lengths, for the comparison.
digest = hashlib.sha256()
for row in unified["test"]:
    digest.update((json.dumps(row, sort_keys=True, ensure_ascii=False) + "\n").encode("utf-8"))
TEST_HASH = digest.hexdigest()
save_json(DATA_DIR / "test_sources.json", test_sources)
save_json(DATA_DIR / "run_manifest.json", {
    "versions": VERSIONS, "model": MODEL_NAME, "seed": SEED, "test_sha256": TEST_HASH,
    "datasets": DATASET_REPOS, "train_caps": TRAIN_CAPS,
    "split_counts": {split: len(tokenized[split]) for split in SPLITS},
    "source_counts": {split: dict(Counter(unified[split]["source"])) for split in SPLITS},
    "max_input_length": MAX_INPUT_LENGTH, "max_target_length": MAX_TARGET_LENGTH,
})


## Shared ROUGE and full-test evaluation
Matches Phase 5: stemming, 0–1 scores, resource-free NLTK Punkt sentence splitting for ROUGE-Lsum, and references decoded from the 128-token labels. Thus references are truncated exactly as in training. Nine qualitative examples and all predictions are saved for each run. Full baseline/post-training generation uses FP32 on the GPU for consistent scoring; FP16 is used during training.

In [ ]:
metric = evaluate.load("rouge")
sentence_tokenizer = PunktSentenceTokenizer()
ROUGE_KEYS = ("rouge1", "rouge2", "rougeL", "rougeLsum")

def compute_scores(predictions, references):
    def sentences(text):
        return "\n".join(sentence_tokenizer.tokenize(text.strip()))
    np.random.seed(SEED)
    scores = metric.compute(predictions=[sentences(text) for text in predictions],
                            references=[sentences(text) for text in references], use_stemmer=True)
    return {key: float(scores[key]) for key in ROUGE_KEYS}

def decode_labels(labels):
    return tokenizer.batch_decode(np.where(np.asarray(labels) == -100,
                                            tokenizer.pad_token_id, labels), skip_special_tokens=True)

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions == -100, tokenizer.pad_token_id, predictions)
    return compute_scores(tokenizer.batch_decode(predictions, skip_special_tokens=True),
                          decode_labels(labels))

def full_test_evaluation(model, label):
    model.eval()
    predictions, references, samples = [], [], []
    sample_counts = Counter()
    started = time.perf_counter()
    with torch.inference_mode():
        for start in tqdm(range(0, len(tokenized["test"]), EVAL_BATCH_SIZE), desc=label):
            batch = tokenized["test"][start:start + EVAL_BATCH_SIZE]
            ids = torch.tensor(batch["input_ids"], dtype=torch.long, device="cuda")
            mask = torch.tensor(batch["attention_mask"], dtype=torch.long, device="cuda")
            width = int(mask.sum(dim=1).max().item())
            output = model.generate(input_ids=ids[:, :width], attention_mask=mask[:, :width],
                                    num_beams=4, max_length=MAX_TARGET_LENGTH,
                                    do_sample=False, use_cache=True)
            pred = tokenizer.batch_decode(output.cpu(), skip_special_tokens=True)
            refs = decode_labels(batch["labels"])
            inputs = tokenizer.batch_decode(batch["input_ids"], skip_special_tokens=True)
            predictions.extend(pred)
            references.extend(refs)
            for offset, (text, prediction, reference) in enumerate(zip(inputs, pred, refs)):
                source = test_sources[start + offset]
                if sample_counts[source] < 3:
                    samples.append({"source": source, "test_index": start + offset,
                                    "input_excerpt": text[:1000], "generated_summary": prediction,
                                    "reference_summary": reference})
                    sample_counts[source] += 1
    assert len(predictions) == len(test_sources)
    # Preserve generated text before potentially lengthy ROUGE computation.
    save_json(DATA_DIR / f"{label}_predictions_full.json",
              [{"test_index": i, "source": source, "prediction": pred, "reference": ref}
               for i, (source, pred, ref) in enumerate(zip(test_sources, predictions, references))])
    by_source = {}
    for source in DATASET_REPOS:
        indices = [i for i, name in enumerate(test_sources) if name == source]
        by_source[source] = compute_scores([predictions[i] for i in indices],
                                            [references[i] for i in indices])
    results = {
        "pooled": compute_scores(predictions, references), "by_source": by_source,
        "model": MODEL_NAME if label == "baseline" else str(BEST_DIR),
        "num_examples": len(predictions), "test_sha256": TEST_HASH,
        "evaluation_config": {"num_beams": 4, "max_length": MAX_TARGET_LENGTH,
                              "seed": SEED, "batch_size": EVAL_BATCH_SIZE, "use_stemmer": True,
                              "sentence_splitter": "nltk.PunktSentenceTokenizer (untrained)",
                              "references": "decoded Phase 3 labels, truncated to 128 tokens"},
        "elapsed_seconds": time.perf_counter() - started,
    }
    save_json(DATA_DIR / f"{label}_results_full.json", results)
    save_json(DATA_DIR / f"{label}_qualitative_samples.json", samples)
    display(pd.DataFrame({"pooled": results["pooled"], **by_source}).T)
    return results


In [ ]:
# Load the unmodified model exactly once. Never rerun this baseline cell after training
# using the trained object; rerunning this cell explicitly reloads the base weights.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to("cuda")
baseline = full_test_evaluation(model, "baseline")
print("Authoritative full baseline saved:", DATA_DIR / "baseline_results_full.json")


## Fine-tune and select the best validation checkpoint
Defaults: batch size 8 × accumulation 2 = effective batch 16, three epochs, FP16. If a T4/P100 runs out of memory, restart from a clean kernel (or resume a saved checkpoint) with train/eval batch sizes 4 and accumulation 4. Gradient checkpointing is enabled to reduce memory use.

If FP16 produces non-finite loss, the guard below stops the run. Set `USE_FP16=False`, reduce the batch size if needed, and restart or resume from a known-good checkpoint. Do not interpret failed or skipped updates as successful training.

Full validation contains about 25.5k rows and is generated at every 500-step evaluation: with 40k training rows and three epochs this is roughly 15 validation passes. Budget GPU time accordingly. Trainer settings reference: [Hugging Face Seq2SeqTrainer](https://huggingface.co/docs/transformers/main_classes/trainer).

In [ ]:
class FiniteLossGuard(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        for key in ("loss", "eval_loss", "grad_norm"):
            value = (logs or {}).get(key)
            if value is not None and not math.isfinite(float(value)):
                raise FloatingPointError(f"Non-finite {key}. Restart with USE_FP16=False; inspect the checkpoint.")

model.config.use_cache = False
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    predict_with_generate=True,
    fp16=USE_FP16,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=3,
    learning_rate=3e-4,
    eval_strategy="steps", eval_steps=500,
    save_strategy="steps", save_steps=500, save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL", greater_is_better=True,
    logging_steps=100, logging_nan_inf_filter=False,
    generation_max_length=MAX_TARGET_LENGTH, generation_num_beams=4,
    gradient_checkpointing=True,
    eval_accumulation_steps=8,
    seed=SEED, data_seed=SEED,
    report_to="none",
)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100),
    compute_metrics=compute_metrics,
    callbacks=[FiniteLossGuard()],
)
train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
assert trainer.state.best_model_checkpoint is not None, "No best checkpoint was selected."
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation ROUGE-L:", trainer.state.best_metric)
# load_best_model_at_end has already restored the selected weights.
model = trainer.model
model.gradient_checkpointing_disable()
model.config.use_cache = True
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))
trainer.save_state()
save_json(DATA_DIR / "training_log_history.json", trainer.state.log_history)
save_json(DATA_DIR / "training_metrics.json", train_result.metrics)
save_json(DATA_DIR / "training_settings.json", training_args.to_dict())
print("Best model and tokenizer saved:", BEST_DIR)


## Training curves
These plots show observed training loss and validation ROUGE-L. Improvement is an empirical outcome, not a guarantee. Full-test before/after scores below provide the held-out comparison.

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
loss = history.dropna(subset=["loss"])
rouge = history.dropna(subset=["eval_rougeL"])
axes[0].plot(loss["step"], loss["loss"], marker=".")
axes[0].set(title="Training loss", xlabel="Optimizer step", ylabel="Loss")
axes[1].plot(rouge["step"], rouge["eval_rougeL"], marker=".")
axes[1].set(title="Validation ROUGE-L", xlabel="Optimizer step", ylabel="ROUGE-L (0–1)")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(DATA_DIR / "training_curves.png", dpi=180)
plt.show()
history.to_csv(DATA_DIR / "training_history.csv", index=False)


In [ ]:
# Release optimizer state before full-test generation; keep the selected model.
del trainer
gc.collect()
torch.cuda.empty_cache()
model.float().to("cuda")
finetuned = full_test_evaluation(model, "finetuned")
assert baseline["num_examples"] == finetuned["num_examples"] == len(test_sources)
assert baseline["test_sha256"] == finetuned["test_sha256"]
assert baseline["evaluation_config"] == finetuned["evaluation_config"]

rows = []
for source in ("pooled", *DATASET_REPOS):
    before = baseline["pooled"] if source == "pooled" else baseline["by_source"][source]
    after = finetuned["pooled"] if source == "pooled" else finetuned["by_source"][source]
    for metric_name in ROUGE_KEYS:
        rows.append({"source": source, "metric": metric_name,
                     "baseline": before[metric_name], "fine_tuned": after[metric_name],
                     "delta": after[metric_name] - before[metric_name]})
comparison = pd.DataFrame(rows)
display(comparison.style.format({"baseline": "{:.4f}", "fine_tuned": "{:.4f}", "delta": "{:+.4f}"}))
comparison.to_csv(DATA_DIR / "baseline_vs_finetuned.csv", index=False)
save_json(DATA_DIR / "baseline_vs_finetuned.json", rows)


## Preserve the outputs
Use Kaggle **Save Version** with outputs retained; after the saved run completes, download from its **Output** panel. Download the best-model archive below, the two `*_results_full.json` reports, qualitative samples, comparison CSV, run manifest, and curve PNG. The `checkpoint-N` directories include Trainer resume state; the compact `best/` export is for inference.

The full GPU baseline supersedes any earlier sampled baseline. Keep `num_examples`, the test hash, library versions, and scoring settings with your portfolio results. No model is uploaded to the Hub by this notebook.

In [ ]:
import shutil
archive = shutil.make_archive(str(CHECKPOINT_DIR.parent / "flan-t5-summarizer-best"),
                              "zip", root_dir=str(BEST_DIR))
print("Download from the saved Kaggle version's Output panel:")
print(archive)
print(DATA_DIR)
print("Save a Kaggle version with outputs before ending the session.")
